# BL4S 2027 — Mahmut Week 2: Geant4 Sensitive Tracker

Corrected public-repository Colab workflow for the Week 2 programming task.

This notebook builds a Geant4 11.4 C++ application, runs an air-control and an aluminium-target condition, records sensitive-detector hits and primary-electron steps, reconstructs approximate directions from six **virtual air scoring planes**, plots a 3D trajectory, and writes/reopens ROOT TTrees with PyROOT.

**Scientific boundary:** the six planes are an educational virtual telescope, not the actual CLEAR detector arrangement. This is not a Timepix3 detector-response simulation or an approved CLEAR setup. The 100 MeV/c momentum and geometry are demonstrator parameters.

In [ ]:
import os
import subprocess
import tempfile
from pathlib import Path

REPO = "MahmutVezir/bl4s-2027-electron-scattering"
BRANCH = "fix/week2-colab-reliability-2026-11-24"

def run_command(command, cwd=None):
    result = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout[-5000:])
    if result.stderr:
        print(result.stderr[-5000:])
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed (exit {result.returncode}): {' '.join(map(str, command))}"
        )
    return result

# Use a fresh clone directory on every run. This avoids stale/partial clones
# left behind when a previous Colab session was interrupted.
REPO_DIR = Path(tempfile.mkdtemp(prefix="bl4s-week2-source-")) / "repo"
clone_url = f"https://github.com/{REPO}.git"
print("Checking public repository access...")
run_command(["git", "--version"])
run_command(["git", "ls-remote", "--exit-code", "--heads", clone_url, BRANCH])
print("Cloning branch:", BRANCH)
run_command(["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH, clone_url, str(REPO_DIR)])

os.chdir(REPO_DIR)
commit = run_command(["git", "rev-parse", "HEAD"], cwd=REPO_DIR).stdout.strip()
active_branch = run_command(["git", "branch", "--show-current"], cwd=REPO_DIR).stdout.strip()
print("Repository:", REPO_DIR)
print("Branch:", active_branch)
print("Commit:", commit)
assert active_branch == BRANCH, f"Unexpected branch: {active_branch}"


## 2. Check source files

In [ ]:
PROJECT = REPO_DIR / "simulation" / "geant4" / "week2"
BUILD = PROJECT / "build"
MACRO = PROJECT / "macros" / "run_demo.mac"

required = [
    PROJECT / "CMakeLists.txt",
    PROJECT / "include" / "TrackerHit.hh",
    PROJECT / "include" / "TrackerSD.hh",
    PROJECT / "src" / "TrackerHit.cc",
    PROJECT / "src" / "TrackerSD.cc",
    PROJECT / "src" / "DetectorConstruction.cc",
    PROJECT / "src" / "PrimaryGeneratorAction.cc",
    PROJECT / "src" / "EventAction.cc",
    PROJECT / "src" / "SteppingAction.cc",
    PROJECT / "src" / "RunAction.cc",
    PROJECT / "src" / "main.cc",
    MACRO,
]
missing = [str(p.relative_to(REPO_DIR)) for p in required if not p.is_file()]
print("Required files:", len(required))
print("Missing:", missing)
if missing:
    raise FileNotFoundError("The Week 2 source package is incomplete.")

## 3. Reuse or install Geant4 11.4.3

A fresh Colab runtime requires internet for the initial installation. If the earlier `/content/geant4-env` environment still exists, this cell reuses it.

In [ ]:
import subprocess
from pathlib import Path

MAMBA = Path("/content/bin/micromamba")
GEANT4_ENV = Path("/content/geant4-env")

if not MAMBA.exists():
    MAMBA.parent.mkdir(parents=True, exist_ok=True)
    bootstrap = subprocess.run(
        "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba",
        shell=True, text=True, capture_output=True
    )
    if bootstrap.returncode != 0 or not MAMBA.exists():
        raise RuntimeError("Micromamba installation failed:\n" + bootstrap.stderr[-3000:])

if not (GEANT4_ENV / "bin" / "geant4-config").exists():
    create = subprocess.run(
        [str(MAMBA), "create", "-y", "-p", str(GEANT4_ENV),
         "-c", "conda-forge", "geant4=11.4.3",
         "geant4-examples=11.4.3", "cmake"],
        text=True, capture_output=True
    )
    print(create.stdout[-5000:])
    print(create.stderr[-3000:])
    if create.returncode != 0:
        raise RuntimeError("Geant4 environment creation failed. Send this full output.")

version = subprocess.run(
    [str(MAMBA), "run", "-p", str(GEANT4_ENV), "geant4-config", "--version"],
    text=True, capture_output=True
)
print("Geant4 version:", version.stdout.strip())
print("Version-check exit code:", version.returncode)
if version.returncode != 0 or not version.stdout.strip().startswith("11.4"):
    raise RuntimeError("Expected a working Geant4 11.4.x environment.")

datasets = subprocess.run(
    [str(MAMBA), "run", "-p", str(GEANT4_ENV), "bash", "-c",
     'env | grep -E "^(G4LEDATA|G4ENSDFSTATEDATA|G4LEVELGAMMADATA|G4NEUTRONHPDATA)=" | sort'],
    text=True, capture_output=True
)
print("\nGeant4 data variables:\n", datasets.stdout)
if datasets.returncode != 0 or "G4LEDATA=" not in datasets.stdout:
    raise RuntimeError("Geant4 data environment variables are not available.")

## 4. Configure CMake and compile

The CMake invocation includes the ZLIB and FreeType paths that were required by the earlier successful B1 build in this Colab environment.

In [ ]:
import subprocess

BUILD.mkdir(parents=True, exist_ok=True)
cmake_command = [
    str(MAMBA), "run", "-p", str(GEANT4_ENV),
    "cmake", "-S", str(PROJECT), "-B", str(BUILD),
    f"-DCMAKE_PREFIX_PATH={GEANT4_ENV}",
    f"-DGeant4_DIR={GEANT4_ENV / 'lib' / 'cmake' / 'Geant4'}",
    f"-DZLIB_ROOT={GEANT4_ENV}",
    f"-DZLIB_INCLUDE_DIR={GEANT4_ENV / 'include'}",
    f"-DZLIB_LIBRARY_RELEASE={GEANT4_ENV / 'lib' / 'libz.so'}",
    f"-DFREETYPE_INCLUDE_DIR_freetype2={GEANT4_ENV / 'include' / 'freetype2'}",
    f"-DFREETYPE_INCLUDE_DIR_ft2build={GEANT4_ENV / 'include' / 'freetype2'}",
    f"-DFREETYPE_LIBRARY_RELEASE={GEANT4_ENV / 'lib' / 'libfreetype.so'}",
    "-DCMAKE_BUILD_TYPE=Release",
]
configure = subprocess.run(cmake_command, text=True, capture_output=True)
print(configure.stdout[-7000:])
print(configure.stderr[-5000:])
print("CMake exit code:", configure.returncode)
if configure.returncode != 0:
    raise RuntimeError("CMake failed. Stop and send the complete CMake output.")

build = subprocess.run(
    [str(MAMBA), "run", "-p", str(GEANT4_ENV),
     "cmake", "--build", str(BUILD), "-j2"],
    cwd=PROJECT, text=True, capture_output=True
)
print(build.stdout[-7000:])
print(build.stderr[-5000:])
print("Build exit code:", build.returncode)

EXECUTABLE = BUILD / "week2_tracker"
if build.returncode != 0 or not EXECUTABLE.is_file():
    raise RuntimeError("C++ compilation failed. Stop and send the full compiler output.")
print("Executable:", EXECUTABLE, "bytes:", EXECUTABLE.stat().st_size)

## 5. Run air-control and aluminium-target conditions

In [ ]:
import os
from datetime import datetime, timezone
import pandas as pd
import numpy as np

RESULTS = REPO_DIR / "results" / "week2_geant4"
RESULTS.mkdir(parents=True, exist_ok=True)
conditions = [
    {"name":"control_air","material":"air"},
    {"name":"aluminium_target","material":"aluminium"},
]
run_records = []

for condition in conditions:
    out_dir = RESULTS / condition["name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    hits_path = out_dir / "hits.csv"
    tracks_path = out_dir / "tracks.csv"
    log_path = out_dir / "simulation.log"

    env = os.environ.copy()
    env["WEEK2_TARGET_MATERIAL"] = condition["material"]
    env["WEEK2_HITS_CSV"] = str(hits_path)
    env["WEEK2_TRACKS_CSV"] = str(tracks_path)

    command = [str(MAMBA), "run", "-p", str(GEANT4_ENV),
               str(EXECUTABLE), str(MACRO)]
    with log_path.open("w") as output:
        result = subprocess.run(
            command, cwd=PROJECT, env=env,
            stdout=output, stderr=subprocess.STDOUT, text=True
        )

    log_text = log_path.read_text(errors="replace")
    print(condition["name"], "exit:", result.returncode)
    print(log_text[-2500:])
    if result.returncode != 0 or "Aborting" in log_text:
        raise RuntimeError(f"Geant4 run failed: {condition['name']}. Send its simulation.log.")
    if not hits_path.is_file() or not tracks_path.is_file():
        raise FileNotFoundError(f"CSV files missing for {condition['name']}.")

    run_records.append({
        "condition":condition["name"],"material":condition["material"],
        "hits_csv":str(hits_path),"tracks_csv":str(tracks_path),
        "log":str(log_path),"exit_code":result.returncode
    })
print("Both conditions completed.")

## 6. Validate sensitive-detector hits and track points

In [ ]:
datasets = {}
track_datasets = {}
hit_columns = [
    "event_id","track_id","parent_id","pdg_code","plane_id",
    "x_mm","y_mm","z_mm","ux","uy","uz",
    "kinetic_energy_MeV","global_time_ns","edep_MeV"
]
track_columns = [
    "event_id","track_id","step_no","x_mm","y_mm","z_mm",
    "kinetic_energy_MeV","edep_MeV","process_name"
]

for item in run_records:
    name = item["condition"]
    hits = pd.read_csv(item["hits_csv"])
    tracks = pd.read_csv(item["tracks_csv"])
    assert set(hit_columns).issubset(hits.columns)
    assert set(track_columns).issubset(tracks.columns)
    assert len(hits)>0 and len(tracks)>0
    assert hits[hit_columns].notna().all().all()
    assert tracks[track_columns].notna().all().all()
    assert (hits["pdg_code"] == 11).all()
    assert (hits["parent_id"] == 0).all()
    assert set(hits["plane_id"].unique()).issubset(set(range(6)))
    datasets[name] = hits
    track_datasets[name] = tracks
    print("\n",name)
    print("Hit rows:",len(hits),"Track-step rows:",len(tracks))
    print("Unique events:",hits["event_id"].nunique())
    print("Events represented per plane:",
          hits.groupby("plane_id")["event_id"].nunique().to_dict())
print("\nCSV checks passed.")

## 7. Reconstruct directions from three upstream and three downstream planes

In [ ]:
def fitted_direction(group):
    z=group["z_mm"].to_numpy(dtype=float)
    x=group["x_mm"].to_numpy(dtype=float)
    y=group["y_mm"].to_numpy(dtype=float)
    if len(np.unique(z))<3:
        return None
    sx=np.polyfit(z,x,1)[0]
    sy=np.polyfit(z,y,1)[0]
    vector=np.array([sx,sy,1.0],dtype=float)
    return vector/np.linalg.norm(vector)

reconstructed={}
for name,hits in datasets.items():
    records=[]
    for event_id,group in hits.groupby("event_id"):
        upstream=group[group["plane_id"].isin([0,1,2])]
        downstream=group[group["plane_id"].isin([3,4,5])]
        if upstream["plane_id"].nunique()!=3 or downstream["plane_id"].nunique()!=3:
            continue
        incoming=fitted_direction(upstream)
        outgoing=fitted_direction(downstream)
        if incoming is None or outgoing is None:
            continue
        cosine=float(np.clip(np.dot(incoming,outgoing),-1.0,1.0))
        records.append({
            "event_id":int(event_id),
            "theta_scatter_deg":float(np.degrees(np.arccos(cosine)))
        })
    reconstructed[name]=pd.DataFrame(records)
    print(name,"fully reconstructed events:",len(records))
if any(frame.empty for frame in reconstructed.values()):
    raise RuntimeError("No complete six-plane tracks. Inspect per-plane event coverage.")

## 8. Plot a primary-electron trajectory in 3D

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def add_plane(ax,z,half=7.04):
    corners=[(-half,-half,z),(half,-half,z),(half,half,z),(-half,half,z)]
    ax.add_collection3d(Poly3DCollection([corners],facecolor="steelblue",alpha=0.12))

track_frame=track_datasets["aluminium_target"]
event_id=int(track_frame["event_id"].iloc[0])
event_track=track_frame[track_frame["event_id"]==event_id].sort_values("step_no")

fig=plt.figure(figsize=(10,7))
ax=fig.add_subplot(111,projection="3d")
for z in [-150,-100,-50,50,100,150]:
    add_plane(ax,z)
v=[(-10,-10,-0.05),(10,-10,-0.05),(10,10,-0.05),(-10,10,-0.05),
   (-10,-10,0.05),(10,-10,0.05),(10,10,0.05),(-10,10,0.05)]
faces=[[v[i] for i in face] for face in
       [(0,1,2,3),(4,5,6,7),(0,1,5,4),(1,2,6,5),(2,3,7,6),(3,0,4,7)]]
ax.add_collection3d(Poly3DCollection(faces,facecolor="orange",alpha=0.45))
ax.plot(event_track["x_mm"],event_track["y_mm"],event_track["z_mm"],linewidth=1.2)
ax.set_xlabel("x (mm)"); ax.set_ylabel("y (mm)"); ax.set_zlabel("z (mm)")
ax.set_title(f"Primary electron trajectory — event {event_id}")
ax.set_box_aspect((1,1,3))
plt.tight_layout()
trajectory_png=RESULTS/"primary_track_3d.png"
plt.savefig(trajectory_png,dpi=180)
plt.show()
print("Saved:",trajectory_png)

## 9. Compare hit maps and reconstructed angles

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
for name,hits in datasets.items():
    last=hits[hits["plane_id"]==5]
    axes[0].scatter(last["x_mm"],last["y_mm"],s=7,alpha=0.35,label=name)
axes[0].set_xlabel("x at virtual plane 6 (mm)")
axes[0].set_ylabel("y at virtual plane 6 (mm)")
axes[0].set_title("Downstream hit maps")
axes[0].legend(); axes[0].grid(alpha=0.2)

for name,frame in reconstructed.items():
    axes[1].hist(frame["theta_scatter_deg"],bins=45,density=True,alpha=0.5,label=name)
axes[1].set_xlabel("Reconstructed angle (degrees)")
axes[1].set_ylabel("Probability density")
axes[1].set_title("Six-plane reconstructed scattering angle")
axes[1].legend(); axes[1].grid(alpha=0.2)
plt.tight_layout()
comparison_png=RESULTS/"control_vs_aluminium.png"
plt.savefig(comparison_png,dpi=180)
plt.show()

summary=pd.DataFrame([
    {"condition":name,"events":len(frame),
     "mean_theta_deg":frame["theta_scatter_deg"].mean(),
     "std_theta_deg":frame["theta_scatter_deg"].std(ddof=1),
     "p95_theta_deg":frame["theta_scatter_deg"].quantile(0.95)}
    for name,frame in reconstructed.items()
])
display(summary)
summary.to_csv(RESULTS/"comparison_summary.csv",index=False)

## 10. Install/check PyROOT

In [ ]:
import importlib.util,sys,subprocess
if importlib.util.find_spec("ROOT") is None:
    install=subprocess.run(
        [sys.executable,"-m","pip","install","root==0.1a12"],
        capture_output=True,text=True
    )
    print(install.stdout[-3000:]); print(install.stderr[-3000:])
    if install.returncode!=0:
        raise RuntimeError("PyROOT installation failed.")
import ROOT
from array import array
ROOT.gROOT.SetBatch(True)
print("ROOT version:",ROOT.gROOT.GetVersion())

## 11. Write Geant4 hit records to ROOT TTrees and histograms

In [ ]:
root_path=RESULTS/"week2_tracker.root"
root_file=ROOT.TFile(str(root_path),"RECREATE")
tree_counts={}

for name,frame in datasets.items():
    tree=ROOT.TTree(f"{name}_hits",f"Geant4 tracker hits: {name}")
    int_fields=["event_id","track_id","parent_id","pdg_code","plane_id"]
    double_fields=["x_mm","y_mm","z_mm","ux","uy","uz",
                   "kinetic_energy_MeV","global_time_ns","edep_MeV"]
    ints={field:array("i",[0]) for field in int_fields}
    doubles={field:array("d",[0.0]) for field in double_fields}
    for field,buffer in ints.items(): tree.Branch(field,buffer,f"{field}/I")
    for field,buffer in doubles.items(): tree.Branch(field,buffer,f"{field}/D")
    for row in frame.to_dict(orient="records"):
        for field in int_fields: ints[field][0]=int(row[field])
        for field in double_fields: doubles[field][0]=float(row[field])
        tree.Fill()
    tree.Write()
    tree_counts[name]=int(tree.GetEntries())

    hx=ROOT.TH1D(f"h_{name}_x_plane6",f"{name};x (mm);Hits",100,-7.04,7.04)
    hy=ROOT.TH1D(f"h_{name}_y_plane6",f"{name};y (mm);Hits",100,-7.04,7.04)
    he=ROOT.TH1D(f"h_{name}_energy",f"{name};Kinetic energy (MeV);Hits",100,90,101)
    for row in frame.to_dict(orient="records"):
        he.Fill(float(row["kinetic_energy_MeV"]))
        if int(row["plane_id"])==5:
            hx.Fill(float(row["x_mm"])); hy.Fill(float(row["y_mm"]))
    hx.Write(); hy.Write(); he.Write()

    ht=ROOT.TH1D(f"h_{name}_theta",f"{name};Scattering angle (degrees);Events",100,0,15)
    for value in reconstructed[name]["theta_scatter_deg"]:
        ht.Fill(float(value))
    ht.Write()

root_file.Close()
print("ROOT file:",root_path,"size:",root_path.stat().st_size)
print("Expected TTree entries:",tree_counts)

In [ ]:
check=ROOT.TFile.Open(str(root_path),"READ")
assert check and not check.IsZombie()
for name,expected in tree_counts.items():
    tree=check.Get(f"{name}_hits")
    assert tree and int(tree.GetEntries())==expected
    theta=check.Get(f"h_{name}_theta")
    assert theta and int(theta.GetEntries())==len(reconstructed[name])
    print(name,"TTree entries:",tree.GetEntries(),"angle entries:",theta.GetEntries())
check.Close()
print("ROOT file reopened and verified.")

## 12. Save metadata and SHA-256 manifest

In [ ]:
import hashlib,json,platform,re,sys
macro_text=MACRO.read_text()
seed_match=re.search(r"/random/setSeeds\s+(\d+)\s+(\d+)",macro_text)
event_match=re.search(r"/run/beamOn\s+(\d+)",macro_text)
g4_version=subprocess.run(
    [str(MAMBA),"run","-p",str(GEANT4_ENV),"geant4-config","--version"],
    capture_output=True,text=True,check=True
).stdout.strip()

metadata={
 "created_utc":datetime.now(timezone.utc).isoformat(),
 "repository":REPO,
 "source_branch":BRANCH,
 "source_commit":subprocess.run(["git","rev-parse","HEAD"],cwd=REPO_DIR,
     capture_output=True,text=True,check=True).stdout.strip(),
 "geant4_version":g4_version,
 "physics_list":"QBBC",
 "primary_particle":"e-",
 "initial_momentum_MeV_c":100.0,
 "target_thickness_mm":0.1,
 "conditions":["air","aluminium"],
 "events_per_condition":int(event_match.group(1)) if event_match else None,
 "random_seeds":[int(seed_match.group(1)),int(seed_match.group(2))] if seed_match else None,
 "python_version":sys.version,
 "platform":platform.platform(),
 "root_version":ROOT.gROOT.GetVersion(),
 "limitations":[
  "Six virtual air scoring planes are not the confirmed CLEAR detector arrangement.",
  "Only primary electrons are included in tracker and trajectory records.",
  "This is not a Timepix3 detector-response simulation.",
  "PyROOT writes the ROOT TTrees from Geant4 CSV output.",
  "Identical random seeds do not guarantee paired event histories."
 ]
}
metadata_path=RESULTS/"experiment_metadata.json"
metadata_path.write_text(json.dumps(metadata,indent=2))

manifest={}
for path in sorted(RESULTS.rglob("*")):
    if path.is_file() and path.name!="SHA256SUMS.json":
        manifest[str(path.relative_to(RESULTS))]={
            "size_bytes":path.stat().st_size,
            "sha256":hashlib.sha256(path.read_bytes()).hexdigest()
        }
manifest_path=RESULTS/"SHA256SUMS.json"
manifest_path.write_text(json.dumps(manifest,indent=2))
print("Metadata:",metadata_path)
print("SHA-256 records:",len(manifest))
print("Manifest:",manifest_path)

## 13. Final validation

In [ ]:
checks={
 "two_runs_completed":len(run_records)==2,
 "hit_data_nonempty":all(len(x)>0 for x in datasets.values()),
 "track_data_nonempty":all(len(x)>0 for x in track_datasets.values()),
 "all_six_planes_seen":all(set(x["plane_id"].unique())==set(range(6)) for x in datasets.values()),
 "six_plane_reconstruction_nonempty":all(len(x)>0 for x in reconstructed.values()),
 "root_file_exists":root_path.is_file() and root_path.stat().st_size>0,
 "metadata_exists":metadata_path.is_file(),
 "sha256_manifest_exists":manifest_path.is_file(),
 "3d_figure_exists":trajectory_png.is_file(),
 "comparison_figure_exists":comparison_png.is_file()
}
for key,value in checks.items(): print("PASS" if value else "FAIL",key)
if not all(checks.values()):
    raise RuntimeError("Some validation checks failed. Save the output and the first traceback.")
print("\nWeek 2 checks passed in this runtime.")

## Optional: full offline transfer package

This is separate from running the simulation. A new Colab runtime still requires internet to start; this package is for restoring the prepared environment on a compatible offline Linux machine. It is disabled by default because the Geant4 environment archive can be large.

In [ ]:
# Set True only when you have enough free disk space.
EXPORT_FULL_OFFLINE_BUNDLE=False
if EXPORT_FULL_OFFLINE_BUNDLE:
    import shutil
    offline=RESULTS/"offline_transfer"
    wheelhouse=offline/"wheelhouse"
    wheelhouse.mkdir(parents=True,exist_ok=True)
    free_gb=shutil.disk_usage("/content").free/(1024**3)
    print(f"Free disk space: {free_gb:.1f} GiB")
    if free_gb<8:
        raise RuntimeError("At least 8 GiB free space is required.")

    subprocess.run([sys.executable,str(REPO_DIR/"tools"/"package_week2_source.py")],check=True)
    source_zip=REPO_DIR/"results"/"week2_source_bundle.zip"
    if source_zip.is_file(): shutil.copy2(source_zip,offline/source_zip.name)

    wheels=subprocess.run(
        [sys.executable,"-m","pip","download",
         "-r",str(REPO_DIR/"requirements-week2-analysis.txt"),"-d",str(wheelhouse)],
        capture_output=True,text=True
    )
    print(wheels.stdout[-3000:]); print(wheels.stderr[-3000:])
    if wheels.returncode!=0: raise RuntimeError("Wheelhouse export failed.")

    subprocess.run([sys.executable,"-m","pip","install","conda-pack"],check=True)
    packer=shutil.which("conda-pack")
    if not packer: raise RuntimeError("conda-pack was not found.")
    env_archive=offline/"geant4-env-11.4.3.tar.gz"
    packed=subprocess.run(
        [packer,"-p",str(GEANT4_ENV),"-o",str(env_archive),"--format","tar.gz"],
        capture_output=True,text=True
    )
    print(packed.stdout[-3000:]); print(packed.stderr[-3000:])
    if packed.returncode!=0: raise RuntimeError("Geant4 environment archive failed.")
    print("Offline transfer archive prepared:",offline)
else:
    print("Skipped full environment archive. See docs/offline_workflow.md.")